## ⚙️ Configuração e Importação
Define os caminhos do Google Drive e carrega os arquivos brutos do TSE para o ambiente de processamento.

In [ ]:

# ==========================================
# 01 - CONFIGURAÇÃO
# ==========================================

import os
import re
import unicodedata
import pandas as pd
from pathlib import Path

# Pasta principal do projeto
BASE_DIR = "/content/drive/MyDrive/Projeto_Eleicao2026"

# ------------------------------------------
# SUBPASTAS
# ------------------------------------------

PASTA_CERT_BR = os.path.join(BASE_DIR, "Cert_Criminal_BR")
PASTA_CERT_MG = os.path.join(BASE_DIR, "Cert_Criminal_MG")

PASTA_PROP_BR = os.path.join(BASE_DIR, "Proposta_BR")
PASTA_PROP_MG = os.path.join(BASE_DIR, "Proposta_MG")

# Fotos do TSE
PASTA_FOTOS_BR = os.path.join(BASE_DIR, "Foto_BR")
PASTA_FOTOS_MG = os.path.join(BASE_DIR, "Foto_MG")

# ------------------------------------------
# ARQUIVOS PRINCIPAIS
# ------------------------------------------

ARQUIVOS = {
    "candidatos_br": os.path.join(BASE_DIR, "consulta_cand_2026_BR.csv"),
    "candidatos_mg": os.path.join(BASE_DIR, "consulta_cand_2026_MG.csv"),

    "complementar_br": os.path.join(
        BASE_DIR, "consulta_cand_complementar_2026_BR.csv"
    ),
    "complementar_mg": os.path.join(
        BASE_DIR, "consulta_cand_complementar_2026_MG.csv"
    ),

    "bens_br": os.path.join(BASE_DIR, "bem_candidato_2026_BR.csv"),
    "bens_mg": os.path.join(BASE_DIR, "bem_candidato_2026_MG.csv"),

    "cassacao_br": os.path.join(BASE_DIR, "motivo_cassacao_2026_BR.csv"),
    "cassacao_mg": os.path.join(BASE_DIR, "motivo_cassacao_2026_MG.csv"),

    # Financeiro
    "receitas_br": os.path.join(
        BASE_DIR, "receitas_candidatos_2026_BR.csv"
    ),
    "receitas_mg": os.path.join(
        BASE_DIR, "receitas_candidatos_2026_MG.csv"
    ),

    "despesas_contratadas_br": os.path.join(
        BASE_DIR, "despesas_contratadas_candidatos_2026_BR.csv"
    ),
    "despesas_contratadas_mg": os.path.join(
        BASE_DIR, "despesas_contratadas_candidatos_2026_MG.csv"
    ),

    "despesas_pagas_br": os.path.join(
        BASE_DIR, "despesas_pagas_2026_BR.csv"
    ),
    "despesas_pagas_mg": os.path.join(
        BASE_DIR, "despesas_pagas_2026_MG.csv"
    ),
}

# ------------------------------------------
# PADRÕES PARA NOMES QUE PODEM VARIAR
# ------------------------------------------
# Se o nome exato não existir, a célula 03 tenta
# encontrar o arquivo pelo padrão.

PADROES_ARQUIVOS = {
    "receitas_br": [
        "receitas_candidatos*2026*BR*.csv",
        "receitas_candidatos*BR*.csv",
    ],
    "receitas_mg": [
        "receitas_candidatos*2026*MG*.csv",
        "receitas_candidatos*MG*.csv",
    ],

    "despesas_contratadas_br": [
        "despesas_contratadas_candidatos*2026*BR*.csv",
        "despesas_contratadas_candidatos*BR*.csv",
    ],
    "despesas_contratadas_mg": [
        "despesas_contratadas_candidatos*2026*MG*.csv",
        "despesas_contratadas_candidatos*MG*.csv",
    ],

    "despesas_pagas_br": [
        "despesas_pagas_candidatos*2026*BR*.csv",
        "despesas_pagas*2026*BR*.csv",
        "despesas_pagas*BR*.csv",
    ],
    "despesas_pagas_mg": [
        "despesas_pagas_candidatos*2026*MG*.csv",
        "despesas_pagas*2026*MG*.csv",
        "despesas_pagas*MG*.csv",
    ],
}

print("✓ Configuração carregada.")
print(f"✓ BASE_DIR: {BASE_DIR}")


✓ Configuração carregada.
✓ BASE_DIR: /content/drive/MyDrive/Projeto_Eleicao2026


In [ ]:
!pip install -q -U google-generativeai pypdf

In [ ]:
# ==========================================
# 02 - GOOGLE DRIVE E VALIDAÇÃO DO PROJETO
# ==========================================

from google.colab import drive
import os

drive.mount("/content/drive")

BASE_DIR = "/content/drive/MyDrive/Projeto_Eleicao2026"

if not os.path.isdir(BASE_DIR):
    raise FileNotFoundError(
        f"""
A pasta do projeto não foi encontrada.

Caminho esperado:
{BASE_DIR}

Verifique se a pasta 'Projeto_Eleicao2026'
está diretamente dentro de 'Meu Drive'.
"""
    )

print("✓ Google Drive conectado")
print("✓ Pasta do projeto encontrada")
print(f"\nBASE_DIR = {BASE_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Google Drive conectado
✓ Pasta do projeto encontrada

BASE_DIR = /content/drive/MyDrive/Projeto_Eleicao2026


In [ ]:
# ==========================================
# 02.5 - VERIFICAR ESTRUTURA DO PROJETO
# ==========================================

from pathlib import Path

print("=" * 70)
print("ESTRUTURA DO PROJETO")
print("=" * 70)

base = Path(BASE_DIR)

if not base.exists():
    raise FileNotFoundError(
        f"Pasta não encontrada:\n{BASE_DIR}"
    )

print(f"\n📁 Pasta base:")
print(f"   {BASE_DIR}\n")

print("Conteúdo:")

for item in sorted(base.iterdir(), key=lambda x: (x.is_file(), x.name.lower())):

    if item.is_dir():
        print(f"📁 {item.name}/")

        # Mostra o conteúdo imediato das subpastas
        for subitem in sorted(item.iterdir()):
            if subitem.is_dir():
                print(f"   📁 {subitem.name}/")
            else:
                print(f"   📄 {subitem.name}")

    else:
        print(f"📄 {item.name}")

print("\n" + "=" * 70)
print("FIM DA VERIFICAÇÃO")
print("=" * 70)

ESTRUTURA DO PROJETO

📁 Pasta base:
   /content/drive/MyDrive/Projeto_Eleicao2026

Conteúdo:
📁 Cert_Criminal_BR/
   📄 2026BR280002539825_280017127481. TRF5_1G.pdf.pdf
   📄 2026BR280002539825_280017127482. Ob. Pé - 0805968-77.2022.4.05.8100.pdf.pdf
   📄 2026BR280002539826_280017006169. TRF6_ 1G e 2G_eProc.pdf.pdf
   📄 2026BR280002539826_280017124965. TRF1_1G e 2G.pdf.pdf
   📄 2026BR280002551547_280017125642.pdf.pdf
   📄 2026BR280002551547_280017125644.pdf.pdf
   📄 2026BR280002552487_280017131970.pdf.pdf
   📄 leiame.pdf
📁 Cert_Criminal_MG/
   📄 2026MG130002532924_130016896996.pdf.pdf
   📄 2026MG130002532932_130016876391.pdf.pdf
   📄 2026MG130002532938_130016890752.pdf.pdf
   📄 2026MG130002532938_130017125001.pdf.pdf
   📄 2026MG130002532938_130017125002.pdf.pdf
   📄 2026MG130002532942_130016896081.pdf.pdf
   📄 2026MG130002532944_130017125151.pdf.pdf
   📄 2026MG130002532951_130016995479.pdf.pdf
   📄 2026MG130002533218_130016996029.pdf.pdf
   📄 2026MG130002533219_130016875834.2015.4.01.380

In [ ]:
# ==========================================
# 03 - CARREGAMENTO DOS DADOS
# ==========================================

import pandas as pd
from pathlib import Path

print("=" * 70)
print("CARREGAMENTO DOS DADOS")
print("=" * 70)


# ==================================================
# LOCALIZAR ARQUIVO
# ==================================================

def localizar_arquivo(nome_base):
    """
    Procura primeiro pelo nome exato.
    Caso não encontre, procura por correspondência parcial.
    """

    base = Path(BASE_DIR)

    # Nome exato
    caminho = base / nome_base

    if caminho.exists():
        return caminho

    # Busca parcial
    encontrados = list(base.glob(f"*{nome_base}*"))

    if encontrados:
        return sorted(encontrados)[0]

    return None


# ==================================================
# CARREGAR CSV
# ==================================================

def carregar_csv(nome_base, descricao=None):

    caminho = localizar_arquivo(nome_base)

    if caminho is None:

        print(f"⚠️ {descricao or nome_base}")
        print(f"   Arquivo não encontrado.")

        return pd.DataFrame()

    try:

        df = pd.read_csv(
            caminho,
            sep=";",
            encoding="latin1",
            low_memory=False
        )

    except Exception:

        try:

            df = pd.read_csv(
                caminho,
                sep=";",
                encoding="utf-8-sig",
                low_memory=False
            )

        except Exception as erro:

            print(f"❌ Erro ao carregar:")
            print(f"   {caminho}")
            print(f"   {erro}")

            return pd.DataFrame()

    print(f"✓ {caminho.name}")
    print(f"  Linhas: {len(df):,}")
    print(f"  Colunas: {len(df.columns)}")

    return df


# ==================================================
# CANDIDATOS
# ==================================================

candidatos_br = carregar_csv(
    "consulta_cand_2026_BR.csv"
)

candidatos_mg = carregar_csv(
    "consulta_cand_2026_MG.csv"
)


# ==================================================
# COMPLEMENTAR
# ==================================================

complementar_br = carregar_csv(
    "consulta_cand_complementar_2026_BR.csv"
)

complementar_mg = carregar_csv(
    "consulta_cand_complementar_2026_MG.csv"
)


# ==================================================
# BENS
# ==================================================

bens_br = carregar_csv(
    "bem_candidato_2026_BR.csv"
)

bens_mg = carregar_csv(
    "bem_candidato_2026_MG.csv"
)


# ==================================================
# CASSAÇÃO
# ==================================================

cassacao_br = carregar_csv(
    "motivo_cassacao_2026_BR.csv"
)

cassacao_mg = carregar_csv(
    "motivo_cassacao_2026_MG.csv"
)


# ==================================================
# RECEITAS
# ==================================================

receitas_br = carregar_csv(
    "receitas_candidatos_2026_BRASIL.csv"
)

receitas_mg = carregar_csv(
    "receitas_candidatos_2026_MG.csv"
)


# ==================================================
# DESPESAS CONTRATADAS
# ==================================================

despesas_contratadas_br = carregar_csv(
    "despesas_contratadas_candidatos_2026_BRASIL.csv"
)

despesas_contratadas_mg = carregar_csv(
    "despesas_contratadas_candidatos_2026_MG.csv"
)


# ==================================================
# DESPESAS PAGAS
# ==================================================

despesas_pagas_br = carregar_csv(
    "despesas_pagas_candidatos_2026_BRASIL.csv"
)

despesas_pagas_mg = carregar_csv(
    "despesas_pagas_candidatos_2026_MG.csv"
)


# ==================================================
# DICIONÁRIO CENTRAL
# ==================================================

ARQUIVOS = {

    "candidatos_br": candidatos_br,
    "candidatos_mg": candidatos_mg,

    "complementar_br": complementar_br,
    "complementar_mg": complementar_mg,

    "bens_br": bens_br,
    "bens_mg": bens_mg,

    "cassacao_br": cassacao_br,
    "cassacao_mg": cassacao_mg,

    "receitas_br": receitas_br,
    "receitas_mg": receitas_mg,

    "despesas_contratadas_br": despesas_contratadas_br,
    "despesas_contratadas_mg": despesas_contratadas_mg,

    "despesas_pagas_br": despesas_pagas_br,
    "despesas_pagas_mg": despesas_pagas_mg,
}


# ==================================================
# COMPATIBILIDADE COM AS CÉLULAS ORIGINAIS
# ==================================================

dados = ARQUIVOS.copy()


# ==================================================
# RESUMO
# ==================================================

print("\n" + "=" * 70)
print("CARREGAMENTO CONCLUÍDO")
print("=" * 70)

for nome, df in dados.items():

    if df is None or df.empty:

        print(
            f"{nome:<35} → VAZIO / NÃO ENCONTRADO"
        )

    else:

        print(
            f"{nome:<35} → OK"
        )


print("\n✓ ARQUIVOS criado")
print("✓ dados criado")
print("✓ Variáveis individuais criadas")

CARREGAMENTO DOS DADOS
✓ consulta_cand_2026_BR.csv
  Linhas: 26
  Colunas: 50
✓ consulta_cand_2026_MG.csv
  Linhas: 1,829
  Colunas: 50
✓ consulta_cand_complementar_2026_BR.csv
  Linhas: 26
  Colunas: 49
✓ consulta_cand_complementar_2026_MG.csv
  Linhas: 1,829
  Colunas: 49
✓ bem_candidato_2026_BR.csv
  Linhas: 300
  Colunas: 19
✓ bem_candidato_2026_MG.csv
  Linhas: 6,582
  Colunas: 19
✓ motivo_cassacao_2026_BR.csv
  Linhas: 0
  Colunas: 14
✓ motivo_cassacao_2026_MG.csv
  Linhas: 0
  Colunas: 14
✓ receitas_candidatos_2026_BRASIL.csv
  Linhas: 46,941
  Colunas: 60
✓ receitas_candidatos_2026_MG.csv
  Linhas: 4,167
  Colunas: 60
✓ despesas_contratadas_candidatos_2026_BRASIL.csv
  Linhas: 90,413
  Colunas: 53
✓ despesas_contratadas_candidatos_2026_MG.csv
  Linhas: 6,973
  Colunas: 53
✓ despesas_pagas_candidatos_2026_BRASIL.csv
  Linhas: 31,742
  Colunas: 28
✓ despesas_pagas_candidatos_2026_MG.csv
  Linhas: 3,063
  Colunas: 28

CARREGAMENTO CONCLUÍDO
candidatos_br                       → OK

## 🧹 Limpeza e Normalização
Padroniza os nomes das colunas e remove acentos ou inconsistências textuais para garantir a integridade das buscas.

In [ ]:

# ==========================================
# 04 - NORMALIZAÇÃO
# ==========================================

def normalizar_texto(valor):
    """
    Normaliza texto para pesquisa:
    - remove acentos
    - converte para minúsculo
    - remove espaços duplicados
    """
    if pd.isna(valor):
        return ""

    valor = str(valor).strip().lower()

    valor = unicodedata.normalize("NFKD", valor)
    valor = "".join(
        c for c in valor
        if not unicodedata.combining(c)
    )

    valor = re.sub(r"\s+", " ", valor)

    return valor


def normalizar_colunas(df):
    df = df.copy()

    df.columns = [
        str(col).strip().upper()
        for col in df.columns
    ]

    return df


def normalizar_id(valor):
    """
    Normaliza SQ_CANDIDATO sem transformar em número.
    """
    if pd.isna(valor):
        return ""

    texto = str(valor).strip()

    if texto.lower() in ["nan", "none", "null", ""]:
        return ""

    return texto


def valor_monetario(valor):
    """
    Converte valores monetários do TSE para float.

    Exemplos:
    1234,56     -> 1234.56
    1.234,56    -> 1234.56
    1234.56     -> 1234.56
    """
    if pd.isna(valor):
        return 0.0

    texto = str(valor).strip()

    if texto.upper() in [
        "",
        "#NE",
        "#NULO",
        "NAN",
        "NONE",
        "NULL"
    ]:
        return 0.0

    texto = texto.replace("R$", "").replace(" ", "")

    try:
        if "," in texto:
            # Formato brasileiro: 1.234,56
            texto = texto.replace(".", "")
            texto = texto.replace(",", ".")
        else:
            # Caso tenha mais de um ponto, trata como separador de milhar
            if texto.count(".") > 1:
                texto = texto.replace(".", "")

        return float(texto)

    except Exception:
        return 0.0


def calcular_idade(data_nascimento, referencia=None):
    """
    Calcula idade a partir de DT_NASCIMENTO.
    """
    if pd.isna(data_nascimento):
        return None

    texto = str(data_nascimento).strip()

    if not texto or texto.upper() in ["#NE", "#NULO", "NAN", "NONE"]:
        return None

    formatos = [
        "%d/%m/%Y",
        "%Y-%m-%d",
        "%d-%m-%Y"
    ]

    data = None

    for formato in formatos:
        try:
            data = pd.to_datetime(
                texto,
                format=formato,
                errors="raise"
            )
            break
        except Exception:
            continue

    if data is None:
        try:
            data = pd.to_datetime(
                texto,
                dayfirst=True,
                errors="coerce"
            )
        except Exception:
            return None

    if pd.isna(data):
        return None

    if referencia is None:
        referencia = pd.Timestamp.today()

    idade = referencia.year - data.year

    if (
        (referencia.month, referencia.day)
        < (data.month, data.day)
    ):
        idade -= 1

    return idade


for nome in dados:
    dados[nome] = normalizar_colunas(dados[nome])

print("✓ Dados normalizados.")


✓ Dados normalizados.


## 🔗 Base Unificada
Realiza a fusão dos dados estaduais (MG) e nacionais (BR) em um único DataFrame otimizado para consulta.

In [ ]:
# ==========================================
# 05 - VALIDAÇÃO DAS COLUNAS
# ==========================================

print("=" * 70)
print("VALIDAÇÃO DAS COLUNAS")
print("=" * 70)

# --------------------------------------------------
# Tabelas que devem possuir SQ_CANDIDATO
# --------------------------------------------------

TABELAS_COM_SQ_CANDIDATO = [
    "candidatos_br",
    "candidatos_mg",

    "complementar_br",
    "complementar_mg",

    "bens_br",
    "bens_mg",

    "cassacao_br",
    "cassacao_mg",

    "receitas_br",
    "receitas_mg",

    "despesas_contratadas_br",
    "despesas_contratadas_mg",
]


# --------------------------------------------------
# Tabelas que NÃO possuem SQ_CANDIDATO
# --------------------------------------------------

TABELAS_SEM_SQ_CANDIDATO = [
    "despesas_pagas_br",
    "despesas_pagas_mg",
]


# --------------------------------------------------
# Validação principal
# --------------------------------------------------

for nome in TABELAS_COM_SQ_CANDIDATO:

    df = dados.get(nome)

    if df is None:
        print(f"⚠️ {nome}: tabela não encontrada.")
        continue

    if df.empty:
        print(f"⚠️ {nome}: tabela vazia.")
        continue

    if "SQ_CANDIDATO" in df.columns:
        print(f"✓ {nome}: SQ_CANDIDATO")
    else:
        print(f"❌ {nome}: SQ_CANDIDATO não encontrada.")


# --------------------------------------------------
# Validação das despesas pagas
# --------------------------------------------------

print("\n" + "-" * 70)
print("TABELAS DE DESPESAS PAGAS")
print("-" * 70)

for nome in TABELAS_SEM_SQ_CANDIDATO:

    df = dados.get(nome)

    if df is None:
        print(f"⚠️ {nome}: tabela não encontrada.")
        continue

    if df.empty:
        print(f"⚠️ {nome}: tabela vazia.")
        continue

    if "SQ_PRESTADOR_CONTAS" in df.columns:
        print(
            f"✓ {nome}: usa SQ_PRESTADOR_CONTAS "
            f"(não possui SQ_CANDIDATO)"
        )
    else:
        print(
            f"❌ {nome}: SQ_PRESTADOR_CONTAS também não encontrada."
        )


# --------------------------------------------------
# Descobrir automaticamente as colunas principais
# --------------------------------------------------

def encontrar_coluna(df, candidatos):

    for coluna in candidatos:
        if coluna in df.columns:
            return coluna

    return None


# --------------------------------------------------
# Candidatos
# --------------------------------------------------

df_candidatos = dados["candidatos_mg"]

COLUNA_NOME = encontrar_coluna(
    df_candidatos,
    [
        "NM_CANDIDATO",
        "NM_URNA_CANDIDATO",
        "NM_SOCIAL_CANDIDATO"
    ]
)

COLUNA_ID = encontrar_coluna(
    df_candidatos,
    [
        "SQ_CANDIDATO"
    ]
)


# --------------------------------------------------
# Resultado
# --------------------------------------------------

print("\n" + "=" * 70)

if COLUNA_NOME:
    print(f"Coluna do nome: {COLUNA_NOME}")
else:
    print("❌ Coluna do nome não encontrada.")

if COLUNA_ID:
    print(f"Coluna do ID: {COLUNA_ID}")
else:
    print("❌ Coluna do ID não encontrada.")

print("=" * 70)

VALIDAÇÃO DAS COLUNAS
✓ candidatos_br: SQ_CANDIDATO
✓ candidatos_mg: SQ_CANDIDATO
✓ complementar_br: SQ_CANDIDATO
✓ complementar_mg: SQ_CANDIDATO
✓ bens_br: SQ_CANDIDATO
✓ bens_mg: SQ_CANDIDATO
⚠️ cassacao_br: tabela vazia.
⚠️ cassacao_mg: tabela vazia.
✓ receitas_br: SQ_CANDIDATO
✓ receitas_mg: SQ_CANDIDATO
✓ despesas_contratadas_br: SQ_CANDIDATO
✓ despesas_contratadas_mg: SQ_CANDIDATO

----------------------------------------------------------------------
TABELAS DE DESPESAS PAGAS
----------------------------------------------------------------------
✓ despesas_pagas_br: usa SQ_PRESTADOR_CONTAS (não possui SQ_CANDIDATO)
✓ despesas_pagas_mg: usa SQ_PRESTADOR_CONTAS (não possui SQ_CANDIDATO)

Coluna do nome: NM_CANDIDATO
Coluna do ID: SQ_CANDIDATO


In [ ]:

# ==========================================
# 06 - BASE UNIFICADA DE CANDIDATOS
# ==========================================

def preparar_candidatos(df, uf):
    if df.empty:
        return pd.DataFrame()

    df = df.copy()

    if "SQ_CANDIDATO" not in df.columns:
        raise ValueError(
            f"{uf}: SQ_CANDIDATO não encontrado. "
            f"Colunas disponíveis: {list(df.columns)}"
        )

    coluna_nome_oficial = encontrar_coluna(
        df,
        ["NM_CANDIDATO"]
    )

    coluna_nome_urna = encontrar_coluna(
        df,
        ["NM_URNA_CANDIDATO"]
    )

    if coluna_nome_oficial is None and coluna_nome_urna is None:
        raise ValueError(
            f"{uf}: nenhuma coluna de nome encontrada."
        )

    # ID oficial
    df["_ID_CANDIDATO"] = (
        df["SQ_CANDIDATO"]
        .apply(normalizar_id)
    )

    # UF
    df["_UF"] = uf

    # Nome oficial
    if coluna_nome_oficial:
        df["_NOME_OFICIAL"] = (
            df[coluna_nome_oficial]
            .fillna("")
            .astype(str)
            .str.strip()
        )
    else:
        df["_NOME_OFICIAL"] = ""

    # Nome de urna
    if coluna_nome_urna:
        df["_NOME_URNA"] = (
            df[coluna_nome_urna]
            .fillna("")
            .astype(str)
            .str.strip()
        )
    else:
        df["_NOME_URNA"] = ""


    # Federação (coligação partidária nacional)
    coluna_federacao = encontrar_coluna(
        df,
        ["NM_FEDERACAO", "SG_FEDERACAO"]
    )

    if coluna_federacao:
        df["_FEDERACAO"] = (
            df[coluna_federacao]
            .fillna("")
            .astype(str)
            .str.strip()
        )
    else:
        df["_FEDERACAO"] = ""

    # Estado civil
    coluna_estado_civil = encontrar_coluna(
        df,
        ["DS_ESTADO_CIVIL"]
    )

    if coluna_estado_civil:
        df["_ESTADO_CIVIL"] = (
            df[coluna_estado_civil]
            .fillna("")
            .astype(str)
            .str.strip()
        )
    else:
        df["_ESTADO_CIVIL"] = ""

    # Nome social (quando houver)
    coluna_nome_social = encontrar_coluna(
        df,
        ["NM_SOCIAL_CANDIDATO"]
    )

    if coluna_nome_social:
        df["_NOME_SOCIAL"] = (
            df[coluna_nome_social]
            .fillna("")
            .astype(str)
            .str.strip()
        )
    else:
        df["_NOME_SOCIAL"] = ""

    # Índice de busca considera nome completo + nome de urna
    df["_NOME_BUSCA"] = (
        df["_NOME_OFICIAL"].apply(normalizar_texto)
        + " "
        + df["_NOME_URNA"].apply(normalizar_texto)
    ).str.strip()

    # Chave composta para evitar qualquer colisão entre UF/ID
    df["_CHAVE_CANDIDATO"] = (
        df["_UF"].astype(str)
        + "_"
        + df["_ID_CANDIDATO"].astype(str)
    )

    return df


candidatos_mg = preparar_candidatos(
    dados["candidatos_mg"],
    "MG"
)

candidatos_br = preparar_candidatos(
    dados["candidatos_br"],
    "BR"
)

candidatos = pd.concat(
    [candidatos_br, candidatos_mg],
    ignore_index=True
)

# Remove apenas registros sem chave válida
candidatos = candidatos[
    candidatos["_ID_CANDIDATO"] != ""
].copy()

print("=" * 70)
print("BASE UNIFICADA")
print("=" * 70)

print(f"Brasil:        {len(candidatos_br):,}")
print(f"Minas Gerais:  {len(candidatos_mg):,}")
print(f"Total:         {len(candidatos):,}")

print(
    f"IDs únicos:    "
    f"{candidatos['_ID_CANDIDATO'].nunique():,}"
)


BASE UNIFICADA
Brasil:        26
Minas Gerais:  1,829
Total:         1,855
IDs únicos:    1,855


## 🔎 Funções de Busca
Agrega lógica para localizar bens declarados, dados complementares e eventuais registros de cassação por ID.

In [ ]:
# ==========================================
# 07 - PESQUISA POR NOME
# ==========================================

def pesquisar_candidatos(nome, uf=None):

    termo = normalizar_texto(nome)

    if not termo:
        return pd.DataFrame()

    resultado = candidatos[
        candidatos["_NOME_BUSCA"].str.contains(
            termo,
            regex=False,
            na=False
        )
    ].copy()

    # Filtro opcional por UF
    if uf:
        resultado = resultado[
            resultado["_UF"] == uf.upper()
        ]

    # Remove duplicidades pelo candidato
    if "_ID_CANDIDATO" in resultado.columns:

        resultado = resultado.drop_duplicates(
            subset=["_ID_CANDIDATO"]
        )

    return resultado

In [ ]:
# ==========================================
# 08 - INTERFACE DE PESQUISA
# ==========================================

import ipywidgets as widgets
from IPython.display import display, clear_output


campo_nome = widgets.Text(
    description="Nome:",
    placeholder="Digite o nome do candidato..."
)

filtro_uf = widgets.Dropdown(
    options=[
        ("Todos", ""),
        ("Minas Gerais", "MG"),
        ("Brasil", "BR")
    ],
    description="UF:"
)

botao = widgets.Button(
    description="Pesquisar",
    button_style="primary"
)

saida = widgets.Output()


def executar_pesquisa(b):

    with saida:
        clear_output()

        termo = campo_nome.value

        if not termo.strip():
            print("Digite um nome para pesquisar.")
            return

        resultado = pesquisar_candidatos(
            termo,
            filtro_uf.value
        )

        if resultado.empty:
            print("Nenhum candidato encontrado.")
            return

        print(
            f"{len(resultado)} candidato(s) encontrado(s).\n"
        )

        colunas = [
            coluna_nome,
            "_UF",
            "_ID_CANDIDATO"
        ]

        display(
            resultado[colunas]
            .reset_index(drop=True)
        )


botao.on_click(executar_pesquisa)


display(
    widgets.VBox([
        campo_nome,
        filtro_uf,
        botao,
        saida
    ])
)

In [ ]:

# ==========================================
# 09 - BUSCAR DADOS DO CANDIDATO
# ==========================================

def buscar_por_id(df, candidato_id):
    """
    Busca exclusivamente por SQ_CANDIDATO.
    """
    if df is None or df.empty:
        return pd.DataFrame()

    if "SQ_CANDIDATO" not in df.columns:
        return pd.DataFrame()

    candidato_id = normalizar_id(candidato_id)

    if not candidato_id:
        return pd.DataFrame()

    ids = (
        df["SQ_CANDIDATO"]
        .apply(normalizar_id)
    )

    return df[ids == candidato_id].copy()


In [ ]:

# ==========================================
# 10 - CONSULTAR BENS
# ==========================================

def consultar_bens(candidato_id, uf):
    chave = "bens_" + str(uf).lower()

    df = dados.get(chave)

    if df is None or df.empty:
        return pd.DataFrame()

    return buscar_por_id(
        df,
        candidato_id
    )


In [ ]:

# ==========================================
# 11 - CONSULTAR DADOS COMPLEMENTARES
# ==========================================

def consultar_complementar(candidato_id, uf):
    chave = "complementar_" + str(uf).lower()

    df = dados.get(chave)

    if df is None or df.empty:
        return pd.DataFrame()

    return buscar_por_id(
        df,
        candidato_id
    )


In [ ]:

# ==========================================
# 12 - CONSULTAR CASSAÇÃO
# ==========================================

def consultar_cassacao(candidato_id, uf):
    chave = "cassacao_" + str(uf).lower()

    df = dados.get(chave)

    if df is None or df.empty:
        return pd.DataFrame()

    return buscar_por_id(
        df,
        candidato_id
    )


## 📑 Indexador de Documentos
Mapeia automaticamente os arquivos PDF de certidões e propostas usando Expressões Regulares (Regex) para extração de IDs.

In [ ]:

# ==========================================
# 13 - ÍNDICE DE DOCUMENTOS
# ==========================================

from pathlib import Path
import re


def extrair_id_valido_nome(nome_arquivo, ids_validos):
    """
    Procura blocos de 12 dígitos no nome do arquivo
    e valida contra os SQ_CANDIDATO existentes.

    Não assume mais que o arquivo tenha obrigatoriamente
    o formato 2026MGxxxxxxxxxxxx.
    """
    if not nome_arquivo:
        return None

    nome = str(nome_arquivo)

    possiveis = re.findall(r"\d{12}", nome)

    encontrados = [
        pid for pid in possiveis
        if pid in ids_validos
    ]

    encontrados = list(dict.fromkeys(encontrados))

    if len(encontrados) == 1:
        return encontrados[0]

    return None


def criar_indice_documentos(pasta, uf, ids_validos):
    registros = []

    if not os.path.exists(pasta):
        return pd.DataFrame(
            columns=[
                "nome_arquivo",
                "caminho",
                "_ID_CANDIDATO",
                "_UF"
            ]
        )

    for arquivo in Path(pasta).rglob("*"):
        if not arquivo.is_file():
            continue

        if arquivo.suffix.lower() not in [
            ".pdf",
            ".txt",
            ".html",
            ".htm"
        ]:
            continue

        id_extraido = extrair_id_valido_nome(
            arquivo.name,
            ids_validos
        )

        registros.append({
            "nome_arquivo": arquivo.name,
            "caminho": str(arquivo),
            "_ID_CANDIDATO": id_extraido,
            "_UF": uf
        })

    return pd.DataFrame(
        registros,
        columns=[
            "nome_arquivo",
            "caminho",
            "_ID_CANDIDATO",
            "_UF"
        ]
    )


ids_validos_br = set(
    candidatos_br["_ID_CANDIDATO"].astype(str)
)
ids_validos_mg = set(
    candidatos_mg["_ID_CANDIDATO"].astype(str)
)

documentos = {
    "cert_br": criar_indice_documentos(
        PASTA_CERT_BR,
        "BR",
        ids_validos_br
    ),

    "cert_mg": criar_indice_documentos(
        PASTA_CERT_MG,
        "MG",
        ids_validos_mg
    ),

    "proposta_br": criar_indice_documentos(
        PASTA_PROP_BR,
        "BR",
        ids_validos_br
    ),

    "proposta_mg": criar_indice_documentos(
        PASTA_PROP_MG,
        "MG",
        ids_validos_mg
    )
}

print("=" * 70)
print("ÍNDICE DE DOCUMENTOS")
print("=" * 70)

for chave, df in documentos.items():
    total = len(df)
    associados = (
        df["_ID_CANDIDATO"].notna().sum()
        if not df.empty
        else 0
    )

    print(
        f"{chave:20} → "
        f"{total:,} arquivos | "
        f"{associados:,} associados"
    )


ÍNDICE DE DOCUMENTOS
cert_br              → 8 arquivos | 7 associados
cert_mg              → 1,659 arquivos | 1,658 associados
proposta_br          → 14 arquivos | 13 associados
proposta_mg          → 12 arquivos | 11 associados


## 💳📷 Novos módulos — Financeiro e Fotos
Integra receitas, despesas, limite de gastos e associa as fotos ao `SQ_CANDIDATO` sem renomear os arquivos originais.

In [ ]:

# ==========================================
# 13.5 - ÍNDICE FINANCEIRO
# ==========================================

def obter_coluna(df, possibilidades):
    if df is None or df.empty:
        return None

    for coluna in possibilidades:
        if coluna in df.columns:
            return coluna

    return None


COLUNAS_RECEITA = [
    "VR_RECEITA"
]

COLUNAS_DESPESA_CONTRATADA = [
    "VR_DESPESA_CONTRATADA"
]

COLUNAS_DESPESA_PAGA = [
    # CORRIGIDO: a tabela despesas_pagas do TSE usa VR_PAGTO_DESPESA.
    # Os nomes antigos (VR_DESPESA_PAGA, VR_DESPESA_PAGAMENTO, VR_PAGO)
    # nao existem na base e faziam a soma cair sempre em 0.
    "VR_PAGTO_DESPESA"
]

COLUNAS_LIMITE_GASTOS = [
    "VR_DESPESA_MAX_CAMPANHA"
]


def somar_coluna_financeira(df, candidato_id, possibilidades):
    if df is None or df.empty:
        return 0.0, 0

    if "SQ_CANDIDATO" not in df.columns:
        return 0.0, 0

    coluna_valor = obter_coluna(
        df,
        possibilidades
    )

    if coluna_valor is None:
        return 0.0, 0

    registros = buscar_por_id(
        df,
        candidato_id
    )

    if registros.empty:
        return 0.0, 0

    valores = registros[coluna_valor].apply(
        valor_monetario
    )

    return float(valores.sum()), len(registros)


def obter_limite_gastos(candidato_id, uf):
    df = dados.get(
        "complementar_" + str(uf).lower()
    )

    if df is None or df.empty:
        return 0.0

    coluna = obter_coluna(
        df,
        COLUNAS_LIMITE_GASTOS
    )

    if coluna is None:
        return 0.0

    registros = buscar_por_id(
        df,
        candidato_id
    )

    if registros.empty:
        return 0.0

    valores = registros[coluna].apply(
        valor_monetario
    )

    # O candidato normalmente aparece uma vez.
    # Se houver mais de um registro, usamos o maior
    # valor válido para não somar limites duplicados.
    return float(valores.max())


# --------------------------------------------------
# CORRIGIDO: despesas_pagas nao possui SQ_CANDIDATO.
# A associacao correta e:
#   SQ_CANDIDATO -> SQ_PRESTADOR_CONTAS -> despesas_pagas
# usando "receitas" (que possui os dois campos) como ponte.
# --------------------------------------------------

_MAPA_PRESTADOR_CANDIDATO_CACHE = {}


def obter_mapa_prestador_candidato(uf):
    """
    Constroi (e armazena em cache) o mapa
    SQ_PRESTADOR_CONTAS -> SQ_CANDIDATO a partir de "receitas",
    unica fonte ja carregada que possui os dois campos.

    Se um mesmo SQ_PRESTADOR_CONTAS aparecer associado a mais de
    um SQ_CANDIDATO (associacao ambigua), ele e descartado do mapa
    em vez de ser associado arbitrariamente.
    """
    uf = str(uf).upper()

    if uf in _MAPA_PRESTADOR_CANDIDATO_CACHE:
        return _MAPA_PRESTADOR_CANDIDATO_CACHE[uf]

    df = dados.get("receitas_" + uf.lower())

    mapa = {}

    if (
        df is not None
        and not df.empty
        and "SQ_CANDIDATO" in df.columns
        and "SQ_PRESTADOR_CONTAS" in df.columns
    ):
        base = df[
            ["SQ_CANDIDATO", "SQ_PRESTADOR_CONTAS"]
        ].dropna().copy()

        base["SQ_CANDIDATO"] = base["SQ_CANDIDATO"].apply(
            normalizar_id
        )

        base["SQ_PRESTADOR_CONTAS"] = base["SQ_PRESTADOR_CONTAS"].apply(
            normalizar_id
        )

        base = base[
            (base["SQ_CANDIDATO"] != "")
            & (base["SQ_PRESTADOR_CONTAS"] != "")
        ]

        candidatos_por_prestador = base.groupby(
            "SQ_PRESTADOR_CONTAS"
        )["SQ_CANDIDATO"].nunique()

        # Mantem apenas prestadores com associacao inequivoca
        # (1 SQ_PRESTADOR_CONTAS -> 1 SQ_CANDIDATO).
        prestadores_unicos = candidatos_por_prestador[
            candidatos_por_prestador == 1
        ].index

        base_unica = base[
            base["SQ_PRESTADOR_CONTAS"].isin(prestadores_unicos)
        ].drop_duplicates("SQ_PRESTADOR_CONTAS")

        mapa = dict(
            zip(
                base_unica["SQ_PRESTADOR_CONTAS"],
                base_unica["SQ_CANDIDATO"]
            )
        )

    _MAPA_PRESTADOR_CANDIDATO_CACHE[uf] = mapa

    return mapa


def somar_despesas_pagas(candidato_id, uf):
    """
    Retorna (total_pago, quantidade, status).

    status:
      "ok"             -> associacao encontrada (mesmo que o total seja R$ 0,00)
      "sem_fonte"      -> despesas_pagas nao encontrada/vazia, ou sem as
                          colunas necessarias (SQ_PRESTADOR_CONTAS / valor pago)
      "sem_associacao" -> nao foi possivel localizar/validar o
                          SQ_PRESTADOR_CONTAS do candidato

    Importante: ausencia de associacao NAO retorna 0.0, e sim None,
    para nao ser exibida como se fosse um "R$ 0,00" confirmado.
    """
    uf = str(uf).upper()

    df = dados.get("despesas_pagas_" + uf.lower())

    if df is None or df.empty:
        return None, 0, "sem_fonte"

    coluna_valor = obter_coluna(
        df,
        COLUNAS_DESPESA_PAGA
    )

    if coluna_valor is None or "SQ_PRESTADOR_CONTAS" not in df.columns:
        return None, 0, "sem_fonte"

    mapa_prestador_candidato = obter_mapa_prestador_candidato(uf)

    candidato_id_norm = normalizar_id(candidato_id)

    prestador_id = None

    for p_id, c_id in mapa_prestador_candidato.items():
        if c_id == candidato_id_norm:
            prestador_id = p_id
            break

    if prestador_id is None:
        return None, 0, "sem_associacao"

    registros = df[
        df["SQ_PRESTADOR_CONTAS"].apply(normalizar_id) == prestador_id
    ]

    if registros.empty:
        # Prestador identificado, mas sem despesas pagas lancadas:
        # aqui sim R$ 0,00 e uma informacao real, nao uma falha de associacao.
        return 0.0, 0, "ok"

    valores = registros[coluna_valor].apply(
        valor_monetario
    )

    return float(valores.sum()), len(registros), "ok"


def consultar_financeiro(candidato_id, uf):
    uf = str(uf).upper()

    total_receita, qtd_receitas = somar_coluna_financeira(
        dados.get("receitas_" + uf.lower()),
        candidato_id,
        COLUNAS_RECEITA
    )

    total_contratado, qtd_contratadas = somar_coluna_financeira(
        dados.get("despesas_contratadas_" + uf.lower()),
        candidato_id,
        COLUNAS_DESPESA_CONTRATADA
    )

    # CORRIGIDO: despesas pagas agora seguem
    # SQ_CANDIDATO -> SQ_PRESTADOR_CONTAS -> despesas_pagas,
    # em vez de tentar buscar SQ_CANDIDATO direto em despesas_pagas.
    total_pago, qtd_pagas, status_pagamento = somar_despesas_pagas(
        candidato_id,
        uf
    )

    limite = obter_limite_gastos(
        candidato_id,
        uf
    )

    return {
        "limite_gastos": limite,
        "total_arrecadado": total_receita,
        "total_contratado": total_contratado,
        "total_pago": total_pago,
        "status_pagamento": status_pagamento,
        "quantidade_receitas": qtd_receitas,
        "quantidade_despesas_contratadas": qtd_contratadas,
        "quantidade_despesas_pagas": qtd_pagas
    }


print("=" * 70)
print("ESTRUTURA FINANCEIRA CARREGADA")
print("=" * 70)

for nome in [
    "receitas_br",
    "receitas_mg",
    "despesas_contratadas_br",
    "despesas_contratadas_mg",
    "despesas_pagas_br",
    "despesas_pagas_mg"
]:
    df = dados.get(nome, pd.DataFrame())

    if df.empty:
        print(f"⚠️ {nome}: vazio / não encontrado")
    else:
        print(
            f"✓ {nome}: "
            f"{len(df):,} registros"
        )


ESTRUTURA FINANCEIRA CARREGADA
✓ receitas_br: 46,941 registros
✓ receitas_mg: 4,167 registros
✓ despesas_contratadas_br: 90,413 registros
✓ despesas_contratadas_mg: 6,973 registros
✓ despesas_pagas_br: 31,742 registros
✓ despesas_pagas_mg: 3,063 registros


In [ ]:

# ==========================================
# 13.6 - ÍNDICE DE FOTOS
# ==========================================

from pathlib import Path
import re


EXTENSOES_FOTO = {
    ".jpg",
    ".jpeg",
    ".png",
    ".webp"
}


def criar_indice_fotos(pasta, uf, ids_validos):
    registros = []

    if not os.path.exists(pasta):
        return pd.DataFrame(
            columns=[
                "_ID_CANDIDATO",
                "_UF",
                "nome_arquivo",
                "caminho"
            ]
        )

    for arquivo in Path(pasta).rglob("*"):
        if not arquivo.is_file():
            continue

        if arquivo.suffix.lower() not in EXTENSOES_FOTO:
            continue

        # Busca qualquer bloco de 12 dígitos
        # e valida contra SQ_CANDIDATO.
        possiveis = re.findall(
            r"\d{12}",
            arquivo.stem
        )

        encontrados = [
            pid for pid in possiveis
            if pid in ids_validos
        ]

        encontrados = list(
            dict.fromkeys(encontrados)
        )

        candidato_id = (
            encontrados[0]
            if len(encontrados) == 1
            else None
        )

        registros.append({
            "_ID_CANDIDATO": candidato_id,
            "_UF": uf,
            "nome_arquivo": arquivo.name,
            "caminho": str(arquivo)
        })

    return pd.DataFrame(
        registros,
        columns=[
            "_ID_CANDIDATO",
            "_UF",
            "nome_arquivo",
            "caminho"
        ]
    )


indice_fotos = pd.concat(
    [
        criar_indice_fotos(
            PASTA_FOTOS_BR,
            "BR",
            ids_validos_br
        ),

        criar_indice_fotos(
            PASTA_FOTOS_MG,
            "MG",
            ids_validos_mg
        )
    ],
    ignore_index=True
)

fotos_associadas = indice_fotos[
    indice_fotos["_ID_CANDIDATO"].notna()
].copy()

print("=" * 70)
print("ÍNDICE DE FOTOS")
print("=" * 70)

print(
    f"Total de imagens encontradas: "
    f"{len(indice_fotos):,}"
)

print(
    f"Fotos associadas a candidatos: "
    f"{len(fotos_associadas):,}"
)

print(
    f"Fotos sem associação: "
    f"{len(indice_fotos) - len(fotos_associadas):,}"
)

if not fotos_associadas.empty:
    print("\nAmostra:")
    display(
        fotos_associadas.head(10)
    )


ÍNDICE DE FOTOS
Total de imagens encontradas: 1,852
Fotos associadas a candidatos: 1,852
Fotos sem associação: 0

Amostra:


,_ID_CANDIDATO,_UF,nome_arquivo,caminho
0,280002551933,BR,FBR280002551933_div.jpg,/content/drive/MyDrive/Projeto_Eleicao2026/Fot...
1,280002553883,BR,FBR280002553883_div.jpg,/content/drive/MyDrive/Projeto_Eleicao2026/Fot...
2,280002539826,BR,FBR280002539826_div.jpg,/content/drive/MyDrive/Projeto_Eleicao2026/Fot...
3,280002548139,BR,FBR280002548139_div.jpg,/content/drive/MyDrive/Projeto_Eleicao2026/Fot...
4,280002552485,BR,FBR280002552485_div.jpg,/content/drive/MyDrive/Projeto_Eleicao2026/Fot...
5,280002540694,BR,FBR280002540694_div.jpg,/content/drive/MyDrive/Projeto_Eleicao2026/Fot...
6,280002552486,BR,FBR280002552486_div.jpg,/content/drive/MyDrive/Projeto_Eleicao2026/Fot...
7,280002542548,BR,FBR280002542548_div.jpg,/content/drive/MyDrive/Projeto_Eleicao2026/Fot...
8,280002553884,BR,FBR280002553884_div.jpg,/content/drive/MyDrive/Projeto_Eleicao2026/Fot...
9,280002551976,BR,FBR280002551976_div.jpg,/content/drive/MyDrive/Projeto_Eleicao2026/Fot...


In [ ]:
# ==========================================
# MÓDULO DE RESUMO DE DOCUMENTOS COM GEMINI
# ==========================================
import google.generativeai as genai
from google.colab import userdata
from pypdf import PdfReader

# Configura a API usando a chave salva nos Secrets do Colab
try:
    api_key = userdata.get('Chave_gemini')
    genai.configure(api_key=api_key)
    # Modelo Flash: ideal para textos longos, rápido e econômico
    modelo_ia = genai.GenerativeModel('gemini-3.5-flash-lite')
except Exception as e:
    modelo_ia = None
    print(f"⚠️ Atenção ao carregar API Key: {e}")

def extrair_texto_pdf(caminho_pdf, limite_paginas=20):
    """Extrai texto das primeiras páginas de um PDF."""
    try:
        reader = PdfReader(caminho_pdf)
        texto = ""
        # Limita páginas para evitar estouro desnecessário em documentos gigantes
        total_pags = min(len(reader.pages), limite_paginas)
        for i in range(total_pags):
            pag_texto = reader.pages[i].extract_text()
            if pag_texto:
                texto += pag_texto + "\n"
        return texto.strip()
    except Exception as err:
        return f"Erro ao ler PDF: {err}"

def gerar_resumo_governo(caminho_pdf):
    """Lê o PDF e pede ao Gemini um resumo estruturado e neutro."""
    if not modelo_ia:
        return "❌ Chave GEMINI_API_KEY não configurada nos Secrets do Colab."

    texto_doc = extrair_texto_pdf(caminho_pdf)
    if not texto_doc or texto_doc.startswith("Erro"):
        return "❌ Não foi possível extrair o texto do arquivo."

    prompt = f"""
    Você é um assistente de dados públicos com foco em síntese informativa e imparcial.
    Analise o texto abaixo extraído do plano/proposta de governo de um candidato e gere um resumo claro.

    Diretrizes:
    1. Seja estritamente neutro e factual (não faça juízo de valor).
    2. Resuma as principais metas em tópicos objetivos: Economia/Emprego, Saúde, Educação, Segurança e Gestão/Infraestrutura.
    3. Mantenha a resposta concisa (máximo de 2 a 3 frases por eixo principal).

    Texto da proposta:
    {texto_doc[:35000]}
    """

    try:
        resposta = modelo_ia.generate_content(prompt)
        return resposta.text
    except Exception as err:
        return f"Erro na chamada da IA: {err}"

print("✅ Módulo de IA carregado com sucesso.")

/usr/local/lib/python3.13/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


✅ Módulo de IA carregado com sucesso.


## 🖥️ Interface Visual e Ficha do Candidato
Dashboard interativo que consolida o perfil completo, patrimônio e análise de propostas com inteligência artificial.

In [ ]:

# ==========================================
# 14 - FICHA COMPLETA DO CANDIDATO
# ==========================================

import html
import ipywidgets as widgets
from IPython.display import display, clear_output, Image as DisplayImage


def formatar_moeda(valor):
    try:
        val_float = float(valor)

        return (
            f"R$ {val_float:,.2f}"
            .replace(",", "X")
            .replace(".", ",")
            .replace("X", ".")
        )

    except Exception:
        return "Não informado"


def limpar_tse(valor, padrao="Não informado"):
    if pd.isna(valor):
        return padrao

    val_str = str(valor).strip().upper()

    if val_str in [
        "#NE",
        "#NULO",
        "-1",
        "-3",
        "NAN",
        "NONE",
        "NULL",
        ""
    ]:
        return padrao

    return str(valor).strip()


def obter_valor_linha(linha, coluna, padrao="Não informado"):
    if coluna not in linha.index:
        return padrao

    return limpar_tse(
        linha.get(coluna),
        padrao
    )


def consultar_foto(candidato_id, uf):
    if indice_fotos.empty:
        return None

    resultado = indice_fotos[
        (indice_fotos["_ID_CANDIDATO"] == str(candidato_id))
        & (indice_fotos["_UF"] == str(uf).upper())
    ]

    if resultado.empty:
        return None

    return resultado.iloc[0]["caminho"]


def consultar_candidato(candidato_id, uf="MG"):
    uf = str(uf).upper()
    candidato_id = str(candidato_id).strip()

    principais = candidatos[
        (candidatos["_ID_CANDIDATO"] == candidato_id)
        & (candidatos["_UF"] == uf)
    ]

    if principais.empty:
        print("❌ Candidato não encontrado.")
        return

    cand = principais.iloc[0]

    # --------------------------------------
    # 1. DADOS BIOGRÁFICOS E POLÍTICOS
    # --------------------------------------

    nome_completo = obter_valor_linha(
        cand,
        "NM_CANDIDATO"
    )

    nome_urna = obter_valor_linha(
        cand,
        "NM_URNA_CANDIDATO"
    )

    numero_urna = obter_valor_linha(
        cand,
        "NR_CANDIDATO"
    )

    cargo = obter_valor_linha(
        cand,
        "DS_CARGO"
    )

    municipio = obter_valor_linha(
        cand,
        "NM_UE"
    )

    partido = limpar_tse(
        cand.get("SG_PARTIDO"),
        "Não informado"
    )

    coligacao = obter_valor_linha(
        cand,
        "NM_COLIGACAO"
    )

    genero = obter_valor_linha(
        cand,
        "DS_GENERO"
    )

    cor_raca = obter_valor_linha(
        cand,
        "DS_COR_RACA"
    )

    instrucao = obter_valor_linha(
        cand,
        "DS_GRAU_INSTRUCAO"
    )

    ocupacao = obter_valor_linha(
        cand,
        "DS_OCUPACAO"
    )

    idade = calcular_idade(
        cand.get("DT_NASCIMENTO")
    )

    idade_texto = (
        f"{idade} anos"
        if idade is not None
        else "Não informado"
    )

    # --------------------------------------
    # 2. SITUAÇÃO
    # --------------------------------------

    situacao = limpar_tse(
        cand.get(
            "DS_DETALHE_SITUACAO_CAND",
            cand.get("DS_SITUACAO_CANDIDATURA")
        ),
        "Aguardando Atualização"
    )

    comp = consultar_complementar(
        candidato_id,
        uf
    )

    status_julgamento = "Não informado"
    tentando_reeleicao = "Não informado"
    limite_gastos_comp = None

    if not comp.empty:
        c_info = comp.iloc[0]

        status_julgamento = obter_valor_linha(
            c_info,
            "DS_SITUACAO_JULGAMENTO"
        )

        valor_reeleicao = c_info.get(
            "ST_REELEICAO"
        )

        if pd.notna(valor_reeleicao):
            texto_reeleicao = str(
                valor_reeleicao
            ).strip().upper()

            if texto_reeleicao in ["S", "SIM", "1", "TRUE"]:
                tentando_reeleicao = "Sim"
            elif texto_reeleicao in ["N", "NAO", "NÃO", "0", "FALSE"]:
                tentando_reeleicao = "Não"
            else:
                tentando_reeleicao = limpar_tse(
                    valor_reeleicao
                )

    # --------------------------------------
    # 3. CABEÇALHO
    # --------------------------------------

    print("=" * 80)
    print(f"👤 {nome_completo}")
    print(f"Nome de urna: {nome_urna}")
    print(
        f"📌 Cargo: {cargo} | "
        f"Número: {numero_urna} | "
        f"Partido: {partido}"
    )
    print(
        f"📍 UF: {uf} | Município: {municipio}"
    )
    print(f"🆔 SQ_CANDIDATO: {candidato_id}")
    print("=" * 80)

    # --------------------------------------
    # 4. FOTO
    # --------------------------------------

    caminho_foto = consultar_foto(
        candidato_id,
        uf
    )

    print("\n📷 FOTO")

    if caminho_foto and os.path.exists(caminho_foto):
        try:
            display(
                DisplayImage(
                    filename=caminho_foto,
                    width=180
                )
            )
            print(
                f"Arquivo: {os.path.basename(caminho_foto)}"
            )
        except Exception as erro:
            print(
                f"Foto encontrada, mas não foi possível "
                f"exibir: {erro}"
            )
    else:
        print("Nenhuma foto associada.")

    # --------------------------------------
    # 5. PERFIL
    # --------------------------------------

    print("\n" + "-" * 80)
    print("📋 DADOS BIOGRÁFICOS E POLÍTICOS")
    print(f"• Nome completo: {nome_completo}")
    print(f"• Nome de urna: {nome_urna}")
    print(f"• Idade: {idade_texto}")
    print(f"• Gênero: {genero}")
    print(f"• Cor/Raça: {cor_raca}")
    print(f"• Instrução: {instrucao}")
    print(f"• Ocupação: {ocupacao}")
    print(f"• Partido: {partido}")
    print(f"• Coligação: {coligacao}")

    # --------------------------------------
    # 6. SITUAÇÃO JURÍDICA/ELEITORAL
    # --------------------------------------

    print("\n" + "-" * 80)
    print("⚖️ SITUAÇÃO JURÍDICA / ELEITORAL")
    print(f"• Situação TSE: {situacao}")
    print(f"• Situação do julgamento: {status_julgamento}")
    print(f"• Tentando reeleição: {tentando_reeleicao}")

    # --------------------------------------
    # 7. PATRIMÔNIO
    # --------------------------------------

    print("\n" + "-" * 80)
    print("💰 PATRIMÔNIO DECLARADO")

    bens = consultar_bens(
        candidato_id,
        uf
    )

    if bens.empty:
        print("Nenhum bem declarado.")
    else:
        bens_clean = bens.copy()

        bens_clean["VALOR_FLOAT"] = (
            bens_clean["VR_BEM_CANDIDATO"]
            .apply(valor_monetario)
        )

        total_patrimonio = float(
            bens_clean["VALOR_FLOAT"].sum()
        )

        print(
            f"Total declarado: "
            f"{formatar_moeda(total_patrimonio)}"
        )

        print(
            f"Quantidade de bens: "
            f"{len(bens_clean)}"
        )

        bens_clean = bens_clean.sort_values(
            by="VALOR_FLOAT",
            ascending=False
        )

        bens_clean["VALOR_FORMATADO"] = (
            bens_clean["VALOR_FLOAT"]
            .apply(formatar_moeda)
        )

        cols_bens = [
            "DS_TIPO_BEM_CANDIDATO",
            "DS_BEM_CANDIDATO",
            "VALOR_FORMATADO"
        ]

        cols_bens = [
            c for c in cols_bens
            if c in bens_clean.columns
        ]

        bens_display = bens_clean[
            cols_bens
        ].rename(
            columns={
                "DS_TIPO_BEM_CANDIDATO": "Tipo",
                "DS_BEM_CANDIDATO": "Descrição",
                "VALOR_FORMATADO": "Valor"
            }
        )

        display(
            bens_display
            .head(15)
            .reset_index(drop=True)
        )

        if len(bens_clean) > 15:
            print(
                f"ℹ️ Exibindo os 15 mais valiosos "
                f"de {len(bens_clean)} bens."
            )

    # --------------------------------------
    # 8. FINANCEIRO
    # --------------------------------------

    print("\n" + "-" * 80)
    print("💳 FINANÇAS DE CAMPANHA")

    financeiro = consultar_financeiro(
        candidato_id,
        uf
    )

    print(
        f"• Limite de gastos: "
        f"{formatar_moeda(financeiro['limite_gastos'])}"
    )

    print(
        f"• Total arrecadado: "
        f"{formatar_moeda(financeiro['total_arrecadado'])}"
    )

    print(
        f"• Total contratado: "
        f"{formatar_moeda(financeiro['total_contratado'])}"
    )

    print(
        f"• Total pago: "
        f"{formatar_moeda(financeiro['total_pago'])}"
    )

    print(
        f"• Registros de receita: "
        f"{financeiro['quantidade_receitas']}"
    )

    print(
        f"• Despesas contratadas: "
        f"{financeiro['quantidade_despesas_contratadas']}"
    )

    print(
        f"• Despesas pagas: "
        f"{financeiro['quantidade_despesas_pagas']}"
    )

    # --------------------------------------
    # 9. CERTIDÕES
    # --------------------------------------

    print("\n" + "-" * 80)
    print("⚖️ CERTIDÕES CRIMINAIS")

    df_cert = documentos.get(
        f"cert_{uf.lower()}"
    )

    if df_cert is None or df_cert.empty:
        print("Pasta de certidões não carregada.")
    else:
        docs = df_cert[
            df_cert["_ID_CANDIDATO"] == candidato_id
        ]

        if docs.empty:
            print(
                "Nenhuma certidão criminal "
                "associada ao candidato."
            )
        else:
            print(
                f"Total de certidões: {len(docs)}"
            )

            for _, doc in docs.iterrows():
                print(
                    f"  📎 {doc['nome_arquivo']}"
                )

    # --------------------------------------
    # 10. CASSAÇÃO
    # --------------------------------------

    print("\n" + "-" * 80)
    print("⚠️ CASSAÇÃO")

    cass = consultar_cassacao(
        candidato_id,
        uf
    )

    if cass.empty:
        print(
            "Nenhum registro de cassação encontrado."
        )
    else:
        print(
            f"Registro(s) encontrado(s): {len(cass)}"
        )

        if "DS_MOTIVO_CASSACAO" in cass.columns:
            for motivo in cass[
                "DS_MOTIVO_CASSACAO"
            ].tolist():
                print(
                    f"• {limpar_tse(motivo)}"
                )

    # --------------------------------------
    # 11. PROPOSTA DE GOVERNO
    # --------------------------------------

    print("\n" + "-" * 80)
    print("📄 PROPOSTA DE GOVERNO")

    df_prop = documentos.get(
        f"proposta_{uf.lower()}"
    )

    if df_prop is None or df_prop.empty:
        print("Pasta de propostas não carregada.")
    else:
        docs = df_prop[
            df_prop["_ID_CANDIDATO"] == candidato_id
        ]

        if docs.empty:
            print(
                "Nenhuma proposta associada."
            )
        else:
            for _, doc in docs.iterrows():
                caminho_arquivo = doc["caminho"]

                print(
                    f"📎 {doc['nome_arquivo']}"
                )

                btn_ia = widgets.Button(
                    description="✨ Gerar Resumo com IA",
                    button_style="warning",
                    icon="magic"
                )

                out_resumo_ia = widgets.Output()

                def disparar_resumo(
                    b,
                    path=caminho_arquivo,
                    out_area=out_resumo_ia
                ):
                    b.disabled = True
                    b.description = "Processando..."

                    with out_area:
                        clear_output()

                        print(
                            "⏳ Lendo e analisando "
                            "a proposta..."
                        )

                        resumo = gerar_resumo_governo(
                            path
                        )

                        clear_output()

                        display(
                            widgets.HTML(
                                f"""
                                <div style="
                                    padding:12px;
                                    border-left:4px solid #f39c12;
                                    margin-top:8px;
                                    border-radius:4px;
                                ">
                                    <b>
                                    Síntese do Plano de Governo:
                                    </b>
                                    <br><br>
                                    {html.escape(resumo).replace(chr(10), "<br>")}
                                </div>
                                """
                            )
                        )

                    b.description = "✔ Concluído"

                btn_ia.on_click(
                    disparar_resumo
                )

                display(
                    widgets.VBox([
                        btn_ia,
                        out_resumo_ia
                    ])
                )

    print("=" * 80)


In [ ]:

# ==========================================
# INTERFACE VISUAL INTEGRADA
# BUSCA POR NOME / UF / CARGO
# ==========================================

import ipywidgets as widgets
from IPython.display import display, clear_output


def pesquisar_candidatos_avancada(
    nome,
    uf=None,
    cargo=None
):
    resultado = candidatos.copy()

    # Nome
    termo = (
        normalizar_texto(nome)
        if nome
        else ""
    )

    if termo:
        resultado = resultado[
            resultado["_NOME_BUSCA"]
            .str.contains(
                termo,
                regex=False,
                na=False
            )
        ]

    # UF
    if uf:
        resultado = resultado[
            resultado["_UF"] == uf.upper()
        ]

    # Cargo
    if cargo:
        if "DS_CARGO" in resultado.columns:
            resultado = resultado[
                resultado["DS_CARGO"]
                .fillna("")
                .astype(str)
                .str.upper()
                == cargo.upper()
            ]

    resultado = resultado.drop_duplicates(
        subset=["_CHAVE_CANDIDATO"]
    )

    return resultado


campo_nome = widgets.Text(
    description="Nome:",
    placeholder="Nome do candidato"
)

filtro_uf = widgets.Dropdown(
    options=[
        ("Todos", ""),
        ("Minas Gerais", "MG"),
        ("Brasil", "BR")
    ],
    description="UF:"
)

opcoes_cargo = [
    ("Todos", ""),
    ("Presidente", "PRESIDENTE"),
    ("Governador", "GOVERNADOR"),
    ("Senador", "SENADOR"),
    ("Deputado Federal", "DEPUTADO FEDERAL"),
    ("Deputado Estadual", "DEPUTADO ESTADUAL")
]

filtro_cargo = widgets.Dropdown(
    options=opcoes_cargo,
    description="Cargo:"
)

btn_pesquisar = widgets.Button(
    description="🔎 Pesquisar",
    button_style="primary"
)

out_resultados = widgets.Output()
out_ficha = widgets.Output()


def ao_clicar_consultar(
    b,
    candidato_id,
    uf
):
    with out_ficha:
        clear_output()
        consultar_candidato(
            candidato_id,
            uf
        )


def executar_pesquisa(b):
    with out_resultados:
        clear_output()

        with out_ficha:
            clear_output()

        termo = campo_nome.value.strip()

        if (
            not termo
            and not filtro_uf.value
            and not filtro_cargo.value
        ):
            print(
                "⚠️ Preencha pelo menos "
                "um filtro."
            )
            return

        resultado = pesquisar_candidatos_avancada(
            termo,
            filtro_uf.value,
            filtro_cargo.value
        )

        if resultado.empty:
            print(
                "Nenhum candidato encontrado."
            )
            return

        if "NM_CANDIDATO" in resultado.columns:
            resultado = resultado.sort_values(
                by="NM_CANDIDATO"
            )

        total_encontrado = len(resultado)

        print(
            f"🔎 {total_encontrado:,} "
            f"candidato(s) encontrado(s).\n"
        )

        if total_encontrado > 50:
            print(
                "⚠️ Exibindo somente os 50 primeiros."
            )

            resultado = resultado.head(50)

        lista_cards = []

        for _, row in resultado.iterrows():
            nome_cand = limpar_tse(
                row.get(
                    "NM_CANDIDATO",
                    row.get("_NOME_OFICIAL")
                )
            )

            uf_cand = row["_UF"]
            id_cand = row["_ID_CANDIDATO"]

            cargo_cand = limpar_tse(
                row.get(
                    "DS_CARGO"
                )
            )

            partido_cand = limpar_tse(
                row.get(
                    "SG_PARTIDO"
                )
            )

            info_html = widgets.HTML(
                value=f"""
                <div style="
                    padding-left:10px;
                    font-size:14px;
                ">
                    <b>{nome_cand}</b><br>
                    {cargo_cand} •
                    {partido_cand} •
                    {uf_cand}<br>
                    SQ_CANDIDATO:
                    {id_cand}
                </div>
                """
            )

            btn_consultar = widgets.Button(
                description="Consultar",
                button_style="info",
                icon="id-card"
            )

            btn_consultar.on_click(
                lambda b,
                cid=id_cand,
                cuf=uf_cand:
                ao_clicar_consultar(
                    b,
                    cid,
                    cuf
                )
            )

            lista_cards.append(
                widgets.HBox([
                    btn_consultar,
                    info_html
                ])
            )

        display(
            widgets.VBox(
                lista_cards
            )
        )


btn_pesquisar.on_click(
    executar_pesquisa
)

display(
    widgets.VBox([
        widgets.HBox([
            campo_nome,
            filtro_uf,
            filtro_cargo,
            btn_pesquisar
        ]),
        out_resultados,
        out_ficha
    ])
)


### 🪐**Exportação para Web**

In [ ]:

# ==========================================
# 15 - EXPORTAÇÃO PARA ARQUITETURA WEB
# ==========================================

import json
import os
import shutil
import numpy as np
from pathlib import Path


print("=" * 80)
print("EXPORTAÇÃO PARA O PROJETO WEB")
print("=" * 80)


# ==========================================
# 15.1 - PASTAS
# ==========================================

PASTA_EXPORTACAO = Path(
    "/content/exportacao_web"
)

if PASTA_EXPORTACAO.exists():
    shutil.rmtree(
        PASTA_EXPORTACAO
    )

PASTA_EXPORTACAO.mkdir(
    parents=True
)

PASTA_DATA = (
    PASTA_EXPORTACAO / "data"
)

PASTA_ASSETS = (
    PASTA_EXPORTACAO / "assets"
)

PASTA_FOTOS = (
    PASTA_ASSETS / "fotos"
)

PASTA_DOCS = (
    PASTA_ASSETS / "documentos"
)

PASTA_PROPOSTAS = (
    PASTA_DOCS / "propostas"
)

PASTA_CERTIDOES = (
    PASTA_DOCS / "certidoes"
)

for pasta in [
    PASTA_DATA,
    PASTA_FOTOS / "BR",
    PASTA_FOTOS / "MG",
    PASTA_PROPOSTAS,
    PASTA_CERTIDOES
]:
    pasta.mkdir(
        parents=True,
        exist_ok=True
    )


# ==========================================
# 15.2 - SERIALIZAÇÃO
# ==========================================

def serializar_json(obj):
    if isinstance(obj, np.integer):
        return int(obj)

    if isinstance(obj, np.floating):
        return float(obj)

    if pd.isna(obj):
        return None

    return str(obj)


def valor_limpo(valor):
    if pd.isna(valor):
        return None

    texto = str(valor).strip()

    if texto.lower() in [
        "",
        "nan",
        "none",
        "null",
        "#ne",
        "#nulo"
    ]:
        return None

    return texto


def escrever_json(nome, objeto):
    caminho = PASTA_DATA / nome

    with open(
        caminho,
        "w",
        encoding="utf-8"
    ) as arquivo:
        json.dump(
            objeto,
            arquivo,
            ensure_ascii=False,
            indent=2,
            default=serializar_json
        )

    return caminho


# ==========================================
# 15.3 - LISTA DE BUSCA
# ==========================================

print("\n1/8 - lista_busca.json")

lista_busca = []

for _, cand in candidatos.iterrows():

    id_cand = valor_limpo(
        cand.get("_ID_CANDIDATO")
    )

    uf = valor_limpo(
        cand.get("_UF")
    )

    if not id_cand or not uf:
        continue

    nome_completo = valor_limpo(
        cand.get("NM_CANDIDATO")
    )

    nome_urna = valor_limpo(
        cand.get("NM_URNA_CANDIDATO")
    )

    foto = None

    if not indice_fotos.empty:
        fotos = indice_fotos[
            (indice_fotos["_ID_CANDIDATO"] == id_cand)
            & (indice_fotos["_UF"] == uf)
        ]

        if not fotos.empty:
            foto_nome = fotos.iloc[0][
                "nome_arquivo"
            ]

            foto = (
                f"assets/fotos/{uf}/{foto_nome}"
            )

    lista_busca.append({
        "id": id_cand,
        "chave": f"{uf}_{id_cand}",
        "nome": nome_urna or nome_completo,
        "nomeCompleto": nome_completo,
        "nomeUrna": nome_urna,
        "nomeBusca": valor_limpo(
            cand.get("_NOME_BUSCA")
        ),
        "numeroUrna": valor_limpo(
            cand.get("NR_CANDIDATO")
        ),
        "uf": uf,
        "cargo": valor_limpo(
            cand.get("DS_CARGO")
        ),
        "partido": (
            valor_limpo(
                cand.get("SG_PARTIDO")
            )
            or valor_limpo(
                cand.get("NM_PARTIDO")
            )
        ),
        "foto": foto
    })


escrever_json(
    "lista_busca.json",
    lista_busca
)

print(
    f"✓ {len(lista_busca):,} candidatos"
)


# ==========================================
# 15.4 - CANDIDATOS
# ==========================================

print("\n2/8 - candidatos.json")

candidatos_json = {}

for _, cand in candidatos.iterrows():

    id_cand = valor_limpo(
        cand.get("_ID_CANDIDATO")
    )

    uf = valor_limpo(
        cand.get("_UF")
    )

    if not id_cand or not uf:
        continue

    idade = calcular_idade(
        cand.get("DT_NASCIMENTO")
    )

    comp = consultar_complementar(
        id_cand,
        uf
    )

    status_julgamento = None
    tentando_reeleicao = None

    # CORRIGIDO: inicializadas aqui fora, antes do "if not comp.empty",
    # para nao gerar NameError quando o candidato nao tem registro
    # complementar (comp vazio).
    situacao_prestacao_contas = None
    situacao_urna = None
    nr_processo = None
    situacao_cassacao = None
    situacao_diploma = None
    situacao_eleitoral = None

    if not comp.empty:
        c = comp.iloc[0]

        status_julgamento = valor_limpo(
            c.get("DS_SITUACAO_JULGAMENTO")
        )

        valor_reeleicao = valor_limpo(
            c.get("ST_REELEICAO")
        )

        if valor_reeleicao:
            vr = valor_reeleicao.upper()

            if vr in ["S", "SIM", "1", "TRUE"]:
                tentando_reeleicao = True
            elif vr in ["N", "NAO", "NÃO", "0", "FALSE"]:
                tentando_reeleicao = False
            else:
                tentando_reeleicao = valor_reeleicao

        # Novos campos — situação complementar
        # CORRIGIDO: DS_PRESTACAO_CONTAS / DS_SITUACAO_PRESTACAO_CONTAS
        # nao existem nas bases do TSE carregadas; o campo real e
        # ST_PREST_CONTAS.
        situacao_prestacao_contas = valor_limpo(
            c.get("ST_PREST_CONTAS")
        )

        # CORRIGIDO: campo real e DS_SITUACAO_CANDIDATO_URNA
        # (DS_SITUACAO_URNA nao existe nas bases carregadas).
        situacao_urna = valor_limpo(
            c.get("DS_SITUACAO_CANDIDATO_URNA")
        )

        nr_processo = valor_limpo(
            c.get("NR_PROCESSO")
        )

        situacao_cassacao = valor_limpo(
            c.get("DS_SITUACAO_CASSACAO")
        )

        situacao_diploma = valor_limpo(
            c.get("DS_SITUACAO_DIPLOMA")
        )

        # CORRIGIDO: DS_SITUACAO_ELEITORAL e DS_DECISAO_JUDICIAL nao
        # existem nas bases carregadas; o campo real para a situacao
        # do candidato no pleito e DS_SITUACAO_CANDIDATO_PLEITO.
        situacao_eleitoral = valor_limpo(
            c.get("DS_SITUACAO_CANDIDATO_PLEITO")
        )


    chave = f"{uf}_{id_cand}"

    candidatos_json[chave] = {
        "id": id_cand,
        "uf": uf,

        "numeroUrna": valor_limpo(
            cand.get("NR_CANDIDATO")
        ),

        "nomeUrna": valor_limpo(
            cand.get("NM_URNA_CANDIDATO")
        ),

        "nomeCompleto": valor_limpo(
            cand.get("NM_CANDIDATO")
        ),

        "cargo": valor_limpo(
            cand.get("DS_CARGO")
        ),

        "municipio": valor_limpo(
            cand.get("NM_UE")
        ),

        "partido": (
            valor_limpo(
                cand.get("SG_PARTIDO")
            )
            or valor_limpo(
                cand.get("NM_PARTIDO")
            )
        ),

        "coligacao": valor_limpo(
            cand.get("NM_COLIGACAO")
        ),

        "genero": valor_limpo(
            cand.get("DS_GENERO")
        ),

        "corRaca": valor_limpo(
            cand.get("DS_COR_RACA")
        ),

        "instrucao": valor_limpo(
            cand.get("DS_GRAU_INSTRUCAO")
        ),

        "ocupacao": valor_limpo(
            cand.get("DS_OCUPACAO")
        ),

        "idade": idade,

        "situacao": valor_limpo(
            cand.get(
                "DS_DETALHE_SITUACAO_CAND",
                cand.get(
                    "DS_SITUACAO_CANDIDATURA"
                )
            )
        ),

        "statusJulgamento": status_julgamento,

        "tentandoReeleicao": tentando_reeleicao,

        "federacao": valor_limpo(
            cand.get("_FEDERACAO")
        ),

        "estadoCivil": valor_limpo(
            cand.get("_ESTADO_CIVIL")
        ),

        "nomeSocial": valor_limpo(
            cand.get("_NOME_SOCIAL")
        ),

        "situacaoPrestacaoContas": situacao_prestacao_contas,

        "situacaoUrna": situacao_urna,

        "nrProcesso": nr_processo,

        "situacaoCassacao": situacao_cassacao,

        "situacaoDiploma": situacao_diploma,

        "situacaoEleitoral": situacao_eleitoral,
    }


escrever_json(
    "candidatos.json",
    candidatos_json
)

print(
    f"✓ {len(candidatos_json):,} candidatos"
)


# ==========================================
# 15.5 - PATRIMÔNIO
# ==========================================

print("\n3/8 - patrimonio.json")

patrimonio_json = {}

for _, cand in candidatos.iterrows():

    id_cand = valor_limpo(
        cand.get("_ID_CANDIDATO")
    )

    uf = valor_limpo(
        cand.get("_UF")
    )

    if not id_cand or not uf:
        continue

    bens = consultar_bens(
        id_cand,
        uf
    )

    lista_bens = []

    total = 0.0

    if not bens.empty:

        for _, bem in bens.iterrows():

            valor = valor_monetario(
                bem.get(
                    "VR_BEM_CANDIDATO"
                )
            )

            total += valor

            lista_bens.append({
                "tipo": valor_limpo(
                    bem.get(
                        "DS_TIPO_BEM_CANDIDATO"
                    )
                ),

                "descricao": valor_limpo(
                    bem.get(
                        "DS_BEM_CANDIDATO"
                    )
                ),

                "valor": valor
            })

    lista_bens.sort(
        key=lambda x: x["valor"],
        reverse=True
    )

    patrimonio_json[
        f"{uf}_{id_cand}"
    ] = {
        "total": total,
        "quantidade": len(lista_bens),
        "bens": lista_bens
    }


escrever_json(
    "patrimonio.json",
    patrimonio_json
)

print(
    f"✓ {len(patrimonio_json):,} registros"
)


# ==========================================
# 15.6 - FINANCEIRO
# ==========================================

print("\n4/8 - financeiro.json")

financeiro_json = {}

for _, cand in candidatos.iterrows():

    id_cand = valor_limpo(
        cand.get("_ID_CANDIDATO")
    )

    uf = valor_limpo(
        cand.get("_UF")
    )

    if not id_cand or not uf:
        continue

    financeiro_json[
        f"{uf}_{id_cand}"
    ] = consultar_financeiro(
        id_cand,
        uf
    )

    # Calcula percentual pago do contratado
    chave_fin = f"{uf}_{id_cand}"
    dados_fin = financeiro_json[chave_fin]
    if (
        dados_fin["total_contratado"] > 0
        and dados_fin["total_pago"] is not None
    ):
        dados_fin["percentual_pago"] = round(
            (dados_fin["total_pago"] / dados_fin["total_contratado"]) * 100,
            1
        )
    else:
        dados_fin["percentual_pago"] = None



escrever_json(
    "financeiro.json",
    financeiro_json
)

print(
    f"✓ {len(financeiro_json):,} registros"
)



# ==========================================
# 15.6b - JURÍDICO (CASSAÇÃO)
# ==========================================

print("\n5/8 - juridico.json")

juridico_json = {}

for _, cand in candidatos.iterrows():

    id_cand = valor_limpo(
        cand.get("_ID_CANDIDATO")
    )

    uf = valor_limpo(
        cand.get("_UF")
    )

    if not id_cand or not uf:
        continue

    cass = consultar_cassacao(
        id_cand,
        uf
    )

    motivos = []

    if not cass.empty:
        if "DS_MOTIVO_CASSACAO" in cass.columns:
            motivos = [
                valor_limpo(m)
                for m in cass["DS_MOTIVO_CASSACAO"].tolist()
                if valor_limpo(m)
            ]

        if "DS_MOTIVO" in cass.columns:
            extras = [
                valor_limpo(m)
                for m in cass["DS_MOTIVO"].tolist()
                if valor_limpo(m)
            ]
            motivos.extend(extras)

    juridico_json[f"{uf}_{id_cand}"] = {
        "motivoCassacao": motivos
    }


escrever_json(
    "juridico.json",
    juridico_json
)

print(
    f"✓ {len(juridico_json):,} registros"
)

# ==========================================
# 15.7 - DOCUMENTOS + CÓPIA FÍSICA
# ==========================================

print("\n6/8 - documentos.json")

documentos_json = {}

total_documentos_copiados = 0


for _, doc in pd.concat(
    documentos.values(),
    ignore_index=True
).iterrows():

    id_cand = valor_limpo(
        doc.get("_ID_CANDIDATO")
    )

    uf = valor_limpo(
        doc.get("_UF")
    )

    caminho_origem = valor_limpo(
        doc.get("caminho")
    )

    nome = valor_limpo(
        doc.get("nome_arquivo")
    )

    if not id_cand or not uf or not caminho_origem or not nome:
        continue

    if not os.path.exists(caminho_origem):
        continue

    chave = f"{uf}_{id_cand}"

    if chave not in documentos_json:
        documentos_json[chave] = {
            "propostas": [],
            "certidoes": []
        }

    if str(doc.get("tipo", "")).lower() == "certidao":
        tipo = "certidao"
    elif "proposta" in str(
        doc.get("nome_arquivo", "")
    ).lower():
        tipo = "proposta"
    else:
        # Descobre pela origem do índice
        tipo = None

    # Determina categoria de forma confiável
    caminho_lower = str(caminho_origem).lower()

    if "cert_criminal" in caminho_lower:
        categoria = "certidoes"
        pasta_destino = PASTA_CERTIDOES
    elif "proposta" in caminho_lower:
        categoria = "propostas"
        pasta_destino = PASTA_PROPOSTAS
    else:
        continue

    nome_destino = nome

    # Evita colisão entre arquivos
    destino = pasta_destino / nome_destino

    try:
        shutil.copy2(
            caminho_origem,
            destino
        )
    except Exception:
        continue

    caminho_relativo = str(
        destino.relative_to(
            PASTA_EXPORTACAO
        )
    ).replace("\\", "/")

    documentos_json[chave][categoria].append({
        "nome": nome,
        "caminho": caminho_relativo
    })

    total_documentos_copiados += 1


escrever_json(
    "documentos.json",
    documentos_json
)

print(
    f"✓ {total_documentos_copiados:,} documentos copiados"
)


# ==========================================
# 15.8 - PROPOSTAS
# ==========================================

print("\n7/8 - textos_propostas.json")

propostas_json = {}
total_propostas = 0

for chave, docs in documentos_json.items():

    propostas = docs.get(
        "propostas",
        []
    )

    if not propostas:
        continue

    propostas_json[chave] = []

    for proposta in propostas:

        caminho_relativo = proposta[
            "caminho"
        ]

        caminho_absoluto = (
            PASTA_EXPORTACAO
            / caminho_relativo
        )

        try:
            texto = extrair_texto_pdf(
                str(caminho_absoluto),
                limite_paginas=20
            )
        except Exception as erro:
            texto = (
                f"Erro ao extrair texto: {erro}"
            )

        propostas_json[chave].append({
            "nome": proposta["nome"],
            "arquivo": caminho_relativo,
            "texto": texto
        })

        total_propostas += 1


escrever_json(
    "textos_propostas.json",
    propostas_json
)

print(
    f"✓ {total_propostas:,} propostas processadas"
)


# ==========================================
# 15.9 - FOTOS + CÓPIA FÍSICA
# ==========================================

print("\n8/8 - fotos.json")

fotos_json = {}
total_fotos_copiadas = 0

for _, foto in indice_fotos.iterrows():

    id_cand = valor_limpo(
        foto.get("_ID_CANDIDATO")
    )

    uf = valor_limpo(
        foto.get("_UF")
    )

    origem = valor_limpo(
        foto.get("caminho")
    )

    nome = valor_limpo(
        foto.get("nome_arquivo")
    )

    if not id_cand or not uf or not origem or not nome:
        continue

    if not os.path.exists(origem):
        continue

    destino = (
        PASTA_FOTOS
        / uf
        / nome
    )

    try:
        shutil.copy2(
            origem,
            destino
        )
    except Exception:
        continue

    relativo = str(
        destino.relative_to(
            PASTA_EXPORTACAO
        )
    ).replace("\\", "/")

    chave = f"{uf}_{id_cand}"

    fotos_json[chave] = {
        "arquivo": relativo
    }

    total_fotos_copiadas += 1


escrever_json(
    "fotos.json",
    fotos_json
)

print(
    f"✓ {total_fotos_copiadas:,} fotos copiadas"
)


# ==========================================
# 15.10 - METADADOS
# ==========================================

metadados = {
    "projeto": "Eleicao 2026",

    "totalCandidatos": len(
        lista_busca
    ),

    "totalFotos": total_fotos_copiadas,

    "totalDocumentos": total_documentos_copiados,

    "totalPropostas": total_propostas,

    "arquivos": [
        "data/lista_busca.json",
        "data/candidatos.json",
        "data/patrimonio.json",
        "data/financeiro.json",
        "data/juridico.json",
        "data/documentos.json",
        "data/textos_propostas.json",
        "data/fotos.json"
    ]
}

escrever_json(
    "metadados.json",
    metadados
)


print("\n" + "=" * 80)
print("✅ EXPORTAÇÃO CONCLUÍDA")
print("=" * 80)

print(
    f"\nCandidatos: {len(lista_busca):,}"
)

print(
    f"Fotos:      {total_fotos_copiadas:,}"
)

print(
    f"Documentos: {total_documentos_copiados:,}"
)

print(
    f"Propostas:  {total_propostas:,}"
)

print(
    f"\n📁 Exportação:"
    f"\n{PASTA_EXPORTACAO}"
)

print("\nEstrutura:")

for caminho in sorted(
    PASTA_EXPORTACAO.rglob("*")
):
    if caminho.is_file():
        relativo = caminho.relative_to(
            PASTA_EXPORTACAO
        )
        tamanho_mb = (
            caminho.stat().st_size
            / 1024
            / 1024
        )

        print(
            f"  📄 {relativo}"
            f" — {tamanho_mb:.2f} MB"
        )


EXPORTAÇÃO PARA O PROJETO WEB

1/7 - lista_busca.json
✓ 1,855 candidatos

2/7 - candidatos.json
✓ 1,855 candidatos

3/7 - patrimonio.json
✓ 1,855 registros

4/7 - financeiro.json
✓ 1,855 registros

5/7 - documentos.json
✓ 1,689 documentos copiados

6/7 - propostas.json


✓ 24 propostas processadas

7/7 - fotos.json
✓ 1,852 fotos copiadas

✅ EXPORTAÇÃO CONCLUÍDA

Candidatos: 1,855
Fotos:      1,852
Documentos: 1,689
Propostas:  24

📁 Exportação:
/content/exportacao_web

Estrutura:
  📄 assets/documentos/certidoes/2026BR280002539825_280017127481. TRF5_1G.pdf.pdf — 0.06 MB
  📄 assets/documentos/certidoes/2026BR280002539825_280017127482. Ob. Pé - 0805968-77.2022.4.05.8100.pdf.pdf — 0.03 MB
  📄 assets/documentos/certidoes/2026BR280002539826_280017006169. TRF6_ 1G e 2G_eProc.pdf.pdf — 0.16 MB
  📄 assets/documentos/certidoes/2026BR280002539826_280017124965. TRF1_1G e 2G.pdf.pdf — 0.22 MB
  📄 assets/documentos/certidoes/2026BR280002551547_280017125642.pdf.pdf — 0.04 MB
  📄 assets/documentos/certidoes/2026BR280002551547_280017125644.pdf.pdf — 0.08 MB
  📄 assets/documentos/certidoes/2026BR280002552487_280017131970.pdf.pdf — 0.13 MB
  📄 assets/documentos/certidoes/2026MG130002532924_130016896996.pdf.pdf — 0.02 MB
  📄 assets/documentos/certidoes/2026MG130002532932